# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
# A.1. Data size, column names, data types
print("Kích thước dữ liệu:", df.shape)
print("Thông tin cột và kiểu dữ liệu:")
df.info()

# A.2. Missing values & Duplicate data
print("Dữ liệu thiếu:\\n", df.isnull().sum())
print("Số dòng trùng lặp:", df.duplicated().sum())
df = df.drop_duplicates() # Xóa dòng trùng lặp nếu cần

# A.3. Invalid values
display(df.describe())

# A.4. Create a new column
df['bmi_group'] = pd.cut(df['bmi'], bins=[0, 25, 30, float('inf')], 
                         labels=['Normal', 'Overweight', 'Obese'], right=False)
display(df[['bmi', 'bmi_group']].head())

Kích thước dữ liệu: (1338, 7)
Thông tin cột và kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB
Dữ liệu thiếu:\n age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
Số dòng trùng lặp: 1


,age,bmi,children,charges
count,1337.000000,1337.000000,1337.000000,1337.000000
mean,39.222139,30.663452,1.095737,13279.121487
std,14.044333,6.100468,1.205571,12110.359656
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.290000,0.000000,4746.344000
50%,39.000000,30.400000,1.000000,9386.161300
75%,51.000000,34.700000,2.000000,16657.717450
max,64.000000,53.130000,5.000000,63770.428010


,bmi,bmi_group
0,27.900,Overweight
1,33.770,Obese
2,33.000,Obese
3,22.705,Normal
4,28.880,Overweight


## A.2. Missing values & Duplicate data

In [4]:
# A.2. Missing values & Duplicate data
print("Kích thước dữ liệu:", df.shape)
print("Số giá trị thiếu theo cột:\n", df.isnull().sum())
print("Số dòng trùng lặp:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Kích thước sau khi bỏ dòng trùng:", df.shape)

Kích thước dữ liệu: (1337, 8)
Số giá trị thiếu theo cột:
 age          0
sex          0
bmi          0
children     0
smoker       0
region       0
charges      0
bmi_group    0
dtype: int64
Số dòng trùng lặp: 0
Kích thước sau khi bỏ dòng trùng: (1337, 8)


## A.3. Invalid values

In [5]:
# A.3. Invalid values
numeric_cols = ['age', 'bmi', 'children', 'charges']
print("Mô tả thống kê cho các cột số:\n")
display(df[numeric_cols].describe().T)

print("\nKiểm tra giá trị âm hoặc rỗng:")
invalid = {}
for col in numeric_cols:
    invalid[col] = int(((df[col].isna()) | (df[col] < 0)).sum())
print(pd.Series(invalid))

print("\nPhân bố hợp lệ của các cột phân loại:")
for col in ['sex', 'smoker', 'region']:
    print(f"\n{col}:\n{df[col].value_counts()}")

Mô tả thống kê cho các cột số:



,count,mean,std,min,25%,50%,75%,max
age,1337.0,39.222139,14.044333,18.0000,27.000,39.0000,51.00000,64.00000
bmi,1337.0,30.663452,6.100468,15.9600,26.290,30.4000,34.70000,53.13000
children,1337.0,1.095737,1.205571,0.0000,0.000,1.0000,2.00000,5.00000
charges,1337.0,13279.121487,12110.359656,1121.8739,4746.344,9386.1613,16657.71745,63770.42801



Kiểm tra giá trị âm hoặc rỗng:
age         0
bmi         0
children    0
charges     0
dtype: int64

Phân bố hợp lệ của các cột phân loại:

sex:
sex
male      675
female    662
Name: count, dtype: int64

smoker:
smoker
no     1063
yes     274
Name: count, dtype: int64

region:
region
southeast    364
southwest    325
northwest    324
northeast    324
Name: count, dtype: int64


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [7]:
# A.4. Create a new column
# Normal (<25), Overweight (25-30), Obese (>=30)
df['bmi_group'] = pd.cut(
    df['bmi'],
    bins=[0, 25, 30, np.inf],
    labels=['Normal', 'Overweight', 'Obese'],
    right=False
)

display(df[['bmi', 'bmi_group']].head(10))
print("\nTần suất bmi_group:\n", df['bmi_group'].value_counts().sort_index())

,bmi,bmi_group
0,27.900,Overweight
1,33.770,Obese
2,33.000,Obese
3,22.705,Normal
4,28.880,Overweight
5,25.740,Overweight
6,33.440,Obese
7,27.740,Overweight
8,29.830,Overweight
9,25.840,Overweight



Tần suất bmi_group:
 bmi_group
Normal        245
Overweight    386
Obese         706
Name: count, dtype: int64


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [8]:
# Group 1 — Central Tendency
num_cols = ['age', 'bmi', 'children', 'charges']
summary = df[num_cols].agg(['mean', 'median']).T
summary.columns = ['mean', 'median']
print("Trung bình và trung vị cho các biến số:\n")
display(summary)

print("\nMode cho các biến số:")
for col in num_cols:
    mode_value = df[col].mode().iloc[0]
    print(f"{col}: {mode_value}")

Trung bình và trung vị cho các biến số:



,mean,median
age,39.222139,39.0000
bmi,30.663452,30.4000
children,1.095737,1.0000
charges,13279.121487,9386.1613



Mode cho các biến số:
age: 18
bmi: 32.3
children: 0
charges: 1121.8739


## Group 2 — Dispersion

In [9]:
# Group 2 — Dispersion
num_cols = ['age', 'bmi', 'children', 'charges']
dispersion = pd.DataFrame({
    'min': df[num_cols].min(),
    'max': df[num_cols].max(),
    'std': df[num_cols].std(),
    'variance': df[num_cols].var(),
    'range': df[num_cols].max() - df[num_cols].min(),
    'IQR': df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25),
})
print("Độ phân tán của các biến số:\n")
display(dispersion.T)

Độ phân tán của các biến số:



,age,bmi,children,charges
min,18.000000,15.960000,0.000000,1.121874e+03
max,64.000000,53.130000,5.000000,6.377043e+04
std,14.044333,6.100468,1.205571,1.211036e+04
variance,197.243282,37.215715,1.453402,1.466608e+08
range,46.000000,37.170000,5.000000,6.264855e+04
IQR,24.000000,8.410000,2.000000,1.191137e+04


## Group 3 — Location and Shape

In [18]:
# Group 3 — Location and Shape
for col in ['age', 'bmi', 'charges']:
    print(f"\n=== {col.upper()} ===")
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    median = df[col].median()
    mean = df[col].mean()
    print(f"Q1 = {q1:.2f}, Median = {median:.2f}, Q3 = {q3:.2f}, Mean = {mean:.2f}")
    print(f"Skewness = {df[col].skew():.4f}")
    print(f"Kurtosis = {df[col].kurt():.4f}")



=== AGE ===
Q1 = 27.00, Median = 39.00, Q3 = 51.00, Mean = 39.22
Skewness = 0.0548
Kurtosis = -1.2444

=== BMI ===
Q1 = 26.29, Median = 30.40, Q3 = 34.70, Mean = 30.66
Skewness = 0.2839
Kurtosis = -0.0529

=== CHARGES ===
Q1 = 4746.34, Median = 9386.16, Q3 = 16657.72, Mean = 13279.12
Skewness = 1.5154
Kurtosis = 1.6042


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [12]:
# Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?
smoker_mean = df.groupby('smoker')['charges'].mean()
non_smoker_mean = smoker_mean['no']
smoker_mean_only = smoker_mean['yes']
ratio = smoker_mean_only / non_smoker_mean

print('Chi phí trung bình theo trạng thái hút thuốc:')
display(smoker_mean.round(2))
print(f'\nTỷ lệ: người hút thuốc trả chi phí cao gấp {ratio:.2f} lần người không hút thuốc.')

region_summary = df.groupby(['region', 'smoker'])['charges'].mean().unstack()
print('\nChi phí trung bình theo vùng và trạng thái hút thuốc:')
display(region_summary.round(2))

region_ratio = (region_summary['yes'] / region_summary['no']).sort_values(ascending=False)
print('\nTỷ lệ hút thuốc / không hút thuốc theo từng vùng:')
display(region_ratio.round(2))

print(f'\nĐộ lệch chuẩn của các tỷ lệ vùng: {region_ratio.std():.2f}')
print('Kết luận: Tỷ lệ chênh lệch giữa các vùng không quá khác biệt, nhưng người hút thuốc vẫn luôn có chi phí cao hơn rõ rệt ở mọi vùng.')

Chi phí trung bình theo trạng thái hút thuốc:


smoker
no      8440.66
yes    32050.23
Name: charges, dtype: float64


Tỷ lệ: người hút thuốc trả chi phí cao gấp 3.80 lần người không hút thuốc.

Chi phí trung bình theo vùng và trạng thái hút thuốc:


smoker,no,yes
region,,
northeast,9165.53,29673.54
northwest,8582.47,30192.00
southeast,8032.22,34845.00
southwest,8019.28,32269.06



Tỷ lệ hút thuốc / không hút thuốc theo từng vùng:


region
southeast    4.34
southwest    4.02
northwest    3.52
northeast    3.24
dtype: float64


Độ lệch chuẩn của các tỷ lệ vùng: 0.49
Kết luận: Tỷ lệ chênh lệch giữa các vùng không quá khác biệt, nhưng người hút thuốc vẫn luôn có chi phí cao hơn rõ rệt ở mọi vùng.


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [19]:
# Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?
smoker_df = df[df['smoker'] == 'yes']
non_smoker_df = df[df['smoker'] == 'no']

corr_smoker = smoker_df[['bmi', 'charges']].corr().iloc[0, 1]
corr_non_smoker = non_smoker_df[['bmi', 'charges']].corr().iloc[0, 1]

print(f'Hệ số tương quan BMI - charges ở nhóm hút thuốc: {corr_smoker:.4f}')
print(f'Hệ số tương quan BMI - charges ở nhóm không hút thuốc: {corr_non_smoker:.4f}')

if abs(corr_smoker) > abs(corr_non_smoker):
    print('Kết luận: BMI có tương quan mạnh hơn với chi phí ở nhóm hút thuốc.')
else:
    print('Kết luận: BMI có tương quan mạnh hơn với chi phí ở nhóm không hút thuốc.')



Hệ số tương quan BMI - charges ở nhóm hút thuốc: 0.8065
Hệ số tương quan BMI - charges ở nhóm không hút thuốc: 0.0841
Kết luận: BMI có tương quan mạnh hơn với chi phí ở nhóm hút thuốc.


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [14]:
# Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?
region_avg = df.groupby('region')['charges'].mean().sort_values(ascending=False)
print('Chi phí bảo hiểm trung bình theo vùng:')
display(region_avg.round(2))
print(f'\nVùng có chi phí trung bình cao nhất là: {region_avg.idxmax()} với {region_avg.max():,.2f}')

Chi phí bảo hiểm trung bình theo vùng:


region
southeast    14735.41
northeast    13406.38
northwest    12450.84
southwest    12346.94
Name: charges, dtype: float64


Vùng có chi phí trung bình cao nhất là: southeast với 14,735.41


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [20]:
# Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?
children_summary = df.groupby('children')['charges'].mean()
print('Chi phí trung bình theo số lượng con cái:')
display(children_summary.round(2))

corr_children = df['children'].corr(df['charges'])
print(f'\nHệ số tương quan giữa số con và chi phí: {corr_children:.4f}')

if corr_children > 0:
    print('Kết luận: Số lượng con cái có xu hướng làm chi phí tăng nhẹ, nhưng mức tương quan rất yếu.')
else:
    print('Kết luận: Số lượng con cái không có hướng làm tăng chi phí.')



Chi phí trung bình theo số lượng con cái:


children
0    12384.70
1    12731.17
2    15073.56
3    15355.32
4    13850.66
5     8786.04
Name: charges, dtype: float64


Hệ số tương quan giữa số con và chi phí: 0.0674
Kết luận: Số lượng con cái có xu hướng làm chi phí tăng nhẹ, nhưng mức tương quan rất yếu.


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [ ]:
# Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?
print(f'Hệ số tương quan giữa age và charges: {df["age"].corr(df["charges"]):.4f}')



Hệ số tương quan giữa age và charges: 0.2983


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*

Tổng hợp, chi phí bảo hiểm chịu ảnh hưởng mạnh nhất bởi tình trạng hút thuốc: người hút thuốc trả trung bình 32.050,23 USD, cao gấp khoảng 3,8 lần so với người không hút thuốc. BMI có mối quan hệ rất mạnh với chi phí trong nhóm hút thuốc (hệ số tương quan 0,8065), cho thấy nguy cơ chi phí cao tăng lên rõ rệt khi hút thuốc kết hợp với BMI cao. Vùng southeast có chi phí trung bình cao nhất (14.735,41 USD), trong khi số lượng con cái chỉ có tương quan rất yếu với chi phí (0,0674), nên không phải yếu tố quyết định chính. Tuổi có mối quan hệ tích cực với chi phí nhưng không mạnh bằng yếu tố hút thuốc, và phân bố chi phí có độ lệch phải rõ ràng, phản ánh sự tập trung của các khoản chi phí cao ở một nhóm nhỏ bệnh nhân. Điều này cho thấy chiến lược kiểm soát rủi ro nên ưu tiên vào hút thuốc và đánh giá BMI, thay vì chỉ dựa vào tuổi hoặc số con.